# Lab 8: Text and Time Series Data

In this lab, we will practice working with both text and time series data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Part 1: Text Data

For this section, we will be using a dataset with academic paper reviews (in both Spanish and English) from the UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/410/paper+reviews

Up to this point in the course, we have only used csv or text files for data. For this lab, we will demonstrate how to use a JSON file.

In [ ]:
import json

# Opening JSON file
f = open('reviews.json', encoding="utf8")

# returns JSON object as a dictionary
data = json.load(f)

To access the text reviews and the reviewer score that we want to predict, we will need a nested for loop.

In [ ]:
text = []
target = []

for paper in data['paper']:
    review = paper['review']
    for rev in review:
        if rev['lan'] == 'es':
            text.append(rev['text'])
            target.append(rev['orientation'])

Next, we need to convert the data into a dataframe.

In [ ]:
text_df = pd.concat([pd.DataFrame(text, columns=['Text']), pd.DataFrame(target, columns=['Target'])], axis=1)
text_df.head(10)

In [ ]:
text_df.shape

In [ ]:
text_df.info()

Empty cells are being interpreted as empty strings, not NaN. Need to convert before we can remove them.

In [ ]:
text_df = pd.concat([pd.DataFrame(text, columns=['Text']).replace('', np.nan), pd.DataFrame(target, columns=['Target'])], axis=1)
text_df.head(10)

In [ ]:
text_df.info()

In [ ]:
# Remove rows with missing text
text_df.dropna(inplace=True)
text_df.info()

Now that the data is cleaned, we can split into feature matrix and target vector and implement:
- Vectorizer method
- Machine learning method
- Validation metrics

In [ ]:
# Separate feature matrix and target vector
X = text_df['Text']
y = text_df['Target']

In [ ]:
# Transform text data with CountVectorizer (use default values)
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
vect.fit(X)
X_vect = vect.transform(X)
X_vect.shape

In [ ]:
# Split into training and testing data (10%)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_vect, y, random_state=0)

In [ ]:
# Implement Grid search with StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=287)

param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}

grid = GridSearchCV(LogisticRegression(max_iter=500), param_grid=param_grid, cv=cv, return_train_score=True)

In [ ]:
# Fit grid to training data
grid.fit(X_train,y_train)

In [ ]:
# Print results
print("Best params:\n{}\n".format(grid.best_params_))
print("Best cross-validation train score: {:.2f}".format(grid.cv_results_['mean_train_score'][grid.best_index_]))
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))

We can adjust the parameters of `CountVectorizer` to try to improve the results. Let's try using `min_df=2` and `ngram_range=(1,3)`:

In [ ]:
# Transform text data with CountVectorizer
vect2 = CountVectorizer(min_df=2, ngram_range=(1, 3))
vect2.fit(X)
X_vect2 = vect2.transform(X)
X_vect2.shape

In [ ]:
# Split into training and testing data (10%)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_vect2, y, random_state=0)

In [ ]:
# Implement Grid search with StratifiedKFold and LogisticRegression()
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=287)

param_grid = {'C': [0.01, 0.1, 1.0, 10.0]}

grid = GridSearchCV(LogisticRegression(max_iter=500), param_grid=param_grid, cv=cv, return_train_score=True)

In [ ]:
# Fit grid to training data
grid.fit(X_train,y_train)

In [ ]:
# Print results
print("Best params:\n{}\n".format(grid.best_params_))
print("Best cross-validation train score: {:.2f}".format(grid.cv_results_['mean_train_score'][grid.best_index_]))
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))

Now let's try implementing Tf-idf. Since this method does use data statistics, we need to use a pipeline:

In [ ]:
# Implement pipeline with TfidfVectorizer and LogisticRegression()
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

pipe = make_pipeline(TfidfVectorizer(min_df=2),
                     LogisticRegression())

param_grid = {'logisticregression__C': [0.001, 0.01, 0.1, 1, 10]}

# Initialize grid and fit it to training data
grid = GridSearchCV(pipe, param_grid, cv=cv, return_train_score=True)
grid.fit(X_train, y_train)

In [ ]:
# Print results
print("Best params:\n{}\n".format(grid.best_params_))
print("Best cross-validation training score: {:.2f}".format(grid.cv_results_['mean_train_score'][grid.best_index_]))
print("Best cross-validation score: {:.2f}".format(grid.best_score_))
print("Test-set score: {:.2f}".format(grid.score(X_test, y_test)))

In [ ]:
# Closing file
f.close()

### Things to think about...

1. Why are the validation results so poor? How could we improve them?
1. What challenges are there with using data in a different language?

## Part 2: Time Series Data

In this section, we are using hourly weather data from Environment Canada for September 2024. We will focus on predicting hourly temperature. We will need to convert the timestamp column from string to datetime type.

In [ ]:
# Import temperature data
data = pd.read_csv('Temperature_data.csv', encoding='unicode_escape')
data.head()

In [ ]:
# Convert date into datetime
data['Date'] = pd.to_datetime(data['Date'])

In [ ]:
data.head()

In [ ]:
# Do we have any missing values?
data.info()

In [ ]:
# Fill in missing values if necessary
data.interpolate(inplace=True)
data.info()

In [ ]:
# Split data into feature matrix and target vector
X = data['Date']
y = data['Temp (°C)']
X.head()

In [ ]:
# Extract hours and days from datetime
hours = pd.DataFrame([i.hour for i in X], columns=['Hour'])
days = pd.DataFrame([i.day for i in X], columns=['Day'])

In [ ]:
# Create a dataframe with hours and days data
dates = pd.concat([hours, days], axis=1)
dates.head()

In [ ]:
# Initialize time series split information
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)

split_idx = int(len(X)-(len(X)/4))

In [ ]:
# Function to implement cross-validation and print scores
def cross_validate_model(model, data, y):
    scores = cross_validate(model, data, y, cv=tscv, return_train_score=True)
    
    print("Training accuracy:", scores['train_score'].mean())
    print("Validation accuracy:", scores['test_score'].mean())

In [ ]:
# Function to plot predicted values compared to original data
def plot_model(model, data, X, y):
    # split the given features into a training and a test set
    X_train, X_test = data[:split_idx], data[split_idx:]
    # also split the target array
    y_train, y_test = y[:split_idx], y[split_idx:]

    model.fit(X_train, y_train)
    pred = np.concatenate((model.predict(X_train),model.predict(X_test)),axis=0)
    plt.plot(X,y)
    plt.plot(X, pred)

You will need to test the following methods:
- RandomForestRegressor() and Ridge() on hours/days data
- RandomForestRegressor() and Ridge() on encoded data
- Ridge() on polynomial transformation of encoded data

You will need to call both functions for each test

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_validate

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=0)

cross_validate_model(rf_regressor, dates, y)

In [ ]:
plot_model(rf_regressor, dates, X, y)

In [ ]:
importances = rf_regressor.feature_importances_
importances

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge()
cross_validate_model(ridge, dates, y)

In [ ]:
plot_model(ridge, dates, X, y)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
dates_onehot = enc.fit_transform(dates).toarray()

In [ ]:
cross_validate_model(ridge, dates_onehot, y)

In [ ]:
cross_validate_model(rf_regressor, dates_onehot, y)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly_transformer = PolynomialFeatures(degree=2, interaction_only=True,
                                      include_bias=False)
dates_onehot_poly = poly_transformer.fit_transform(dates_onehot)

In [ ]:
cross_validate_model(ridge, dates_onehot_poly, y)

In [ ]:
plot_model(ridge, dates_onehot_poly, X, y)

### Things to think about...

1. Why are the validation results so poor? How could we improve them?
1. What challenges are there with using highly variable data (like weather data)?